# 05_implicit_rotire
**Implicit + rotire ±15° + întoarcere sus-jos 0.5 = setările din notebook-ul original.**

Copie a `antrenare_yolo_segmentare_din_txt` (segmentare `yolo11m-seg` pe poligoanele calculate cu GrabCut), cu **altă combinație de parametri** la antrenare.
Datele vin din `date/dataset_seg/` (pregătite o singură dată de `00_pregatire_date`), deci toate experimentele sunt comparabile.

Rezultatele: `loguri/05_implicit_rotire.json` (metrici), `loguri/05_implicit_rotire_metrici.csv`, `loguri/05_implicit_rotire_antrenare.log` (fiecare epocă), modelul în `modele/05_implicit_rotire.pt`.
⚠️ Câștigătorul se alege după **validare**; testul se raportează la final, în lucrare.

## 1. Parametrii experimentului

In [ ]:
# ===================== PARAMETRII EXPERIMENTULUI (doar asta diferă între notebook-uri) =====================
import albumentations as A

NUME_EXPERIMENT = "05_implicit_rotire"
DESCRIERE = "Implicit + rotire ±15° + întoarcere sus-jos 0.5 = setările din notebook-ul original."

# Comune pentru toate experimentele (ca în antrenare_yolo_segmentare_din_txt)
MODEL = "yolo11m-seg.pt"
EPOCI, PATIENCE, IMGSZ, BATCH = 150, 30, 640, 8   # dacă primești „CUDA out of memory” → BATCH = 4

# Parametrii model.train(...) – suprascriu valorile de mai sus dacă apar și aici
PARAMETRI = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # culoare: nuanță, saturație, luminozitate
    translate=0.1, scale=0.5,          # mutare și zoom
    fliplr=0.5,                          # oglindire stânga-dreapta
    degrees=15.0, flipud=0.5,           # rotire (±grade) și întoarcere sus-jos
    mosaic=1.0,                          # mozaic (4 poze lipite)
    erasing=0.0, auto_augment=None,       # contează doar la clasificare – oprite la toate
)

# Transformări Albumentations (doar culori/pixeli – dreptunghiurile și poligoanele rămân corecte)
AUGMENTARI = None   # fără transformări Albumentations în plus

## 2. Pregătire (date, GPU, setări)

In [ ]:
import os, glob, json, shutil, time, logging
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml

BAZA = Path(os.getcwd()).resolve()          # folderul experimente_parametri
SEG = BAZA / "date" / "dataset_seg"
RUNS, LOGURI, MODELE = BAZA / "runs", BAZA / "loguri", BAZA / "modele"
for d in (RUNS, LOGURI, MODELE):
    d.mkdir(exist_ok=True)

if not (SEG / "data_seg.yaml").exists():
    raise FileNotFoundError("Lipsesc datele: rulează întâi 00_pregatire_date.ipynb "
                            "(python ruleaza_toate.py îl rulează singur).")

# Dacă folderul a fost mutat, refacem căile din fișierele .yaml
for f in ["data_seg.yaml", "data_seg_test_vechi.yaml", "data_seg_test_nou.yaml"]:
    y = yaml.safe_load(open(SEG / f))
    if y.get("path") != str(SEG):
        y["path"] = str(SEG)
        yaml.safe_dump(y, open(SEG / f, "w"), allow_unicode=True)
nume = yaml.safe_load(open(SEG / "data_seg.yaml"))["names"]

# Fără internet: fontul pentru grafice îl luăm din folder
try:
    from ultralytics.utils import USER_CONFIG_DIR
    if (BAZA / "Arial.ttf").exists() and not (Path(USER_CONFIG_DIR) / "Arial.ttf").exists():
        Path(USER_CONFIG_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copy(BAZA / "Arial.ttf", Path(USER_CONFIG_DIR) / "Arial.ttf")
except Exception:
    pass

import torch, ultralytics
if torch.cuda.is_available():
    DISPOZITIV = 0
    print("GPU:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DISPOZITIV = "mps"; print("GPU Apple (mps)")
else:
    DISPOZITIV = "cpu"; print("⚠️ Nu am găsit GPU – rulează pe procesor (foarte lent).")
print("Ultralytics", ultralytics.__version__, "| experiment:", NUME_EXPERIMENT)

MODEL_START = str(BAZA / MODEL) if (BAZA / MODEL).exists() else MODEL
setari = dict(data=str(SEG / "data_seg.yaml"), epochs=EPOCI, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
              device=DISPOZITIV, seed=0, project=str(RUNS), name=NUME_EXPERIMENT, exist_ok=True)
setari.update(PARAMETRI)                     # parametrii experimentului suprascriu valorile comune
if AUGMENTARI:
    setari["augmentations"] = AUGMENTARI
print({k: v for k, v in setari.items() if k != "augmentations"})
if AUGMENTARI:
    print("Albumentations:", [t.__class__.__name__ for t in AUGMENTARI])

## 3. Antrenare
Dacă placa s-a oprit la jumătate, la repornire continuă de unde a rămas (`last.pt`).

In [ ]:
from ultralytics import YOLO
DOSAR = RUNS / NUME_EXPERIMENT
ULTIM, MARCAJ = DOSAR / "weights" / "last.pt", DOSAR / "_antrenare_gata"

# Log „live” cu rezultatele fiecărei epoci: loguri/<experiment>_antrenare.log
jurnal_live = logging.FileHandler(LOGURI / f"{NUME_EXPERIMENT}_antrenare.log", encoding="utf-8")
logging.getLogger("ultralytics").addHandler(jurnal_live)
t0 = time.time()
try:
    if MARCAJ.exists():
        print("✅ Antrenarea e deja terminată – trec la evaluare.")
    elif ULTIM.exists():
        print("🔄 Reiau antrenarea întreruptă de la", ULTIM)
        try:
            YOLO(str(ULTIM)).train(resume=True, **({"augmentations": AUGMENTARI} if AUGMENTARI else {}))
        except AssertionError as e:          # „nothing to resume” = se terminase deja
            if "nothing to resume" not in str(e).lower():
                raise
    else:
        YOLO(MODEL_START).train(**setari)
    MARCAJ.write_text(time.strftime("%Y-%m-%d %H:%M"))
finally:
    logging.getLogger("ultralytics").removeHandler(jurnal_live)
    jurnal_live.close()
print(f"Antrenare: {(time.time() - t0) / 60:.1f} min")

## 4. Graficele antrenării

In [ ]:
from IPython.display import Image, display
for f in ["results.png", "confusion_matrix_normalized.png", "MaskPR_curve.png"]:
    if (DOSAR / f).exists():
        display(Image(str(DOSAR / f), width=900))

## 5. Evaluare pe validare și pe test (Box = găsește obiectul, Mask = potrivirea conturului)

In [ ]:
model = YOLO(str(DOSAR / "weights" / "best.pt"))
seturi = {"validare": ("data_seg.yaml", "val"), "test_complet": ("data_seg.yaml", "test"),
          "test_vechi": ("data_seg_test_vechi.yaml", "test"), "test_nou": ("data_seg_test_nou.yaml", "test")}
fisier_lista = {"test_vechi": "test_vechi.txt", "test_nou": "test_nou.txt"}

def f3(x):
    return round(float(x), 4)

metrici, randuri = {}, []
for nume_set, (fisier, split) in seturi.items():
    lista = fisier_lista.get(nume_set)
    if lista and not open(SEG / lista).read().strip():
        continue                                   # ex. fără date noi → nu există test_nou
    r = model.val(data=str(SEG / fisier), split=split, imgsz=setari["imgsz"], batch=setari["batch"],
                  device=DISPOZITIV, verbose=False, plots=False,
                  project=str(RUNS / "_evaluari"), name=f"{NUME_EXPERIMENT}_{nume_set}", exist_ok=True)
    metrici[nume_set] = {}
    for tip, m in [("box", r.box)] + ([("mask", r.seg)] if hasattr(r, "seg") else []):
        pe_clase = {nume[int(c)]: f3(m.ap50[i]) for i, c in enumerate(r.ap_class_index)}
        metrici[nume_set][tip] = {"mAP50": f3(m.map50), "mAP50-95": f3(m.map), "P": f3(m.mp), "R": f3(m.mr),
                                  "mAP50_pe_clase": pe_clase}
        randuri.append({"set": nume_set, "tip": tip, "mAP50": f3(m.map50), "mAP50-95": f3(m.map),
                        "P": f3(m.mp), "R": f3(m.mr), **pe_clase})
tabel = pd.DataFrame(randuri).set_index(["set", "tip"])
display(tabel)
tabel.to_csv(LOGURI / f"{NUME_EXPERIMENT}_metrici.csv")

## 6. Cât de mult „mint” dreptunghiurile despre suprafață? (mediana arie contur / arie dreptunghi, pe test)

In [ ]:
rapoarte = {}
for cale in glob.glob(str(SEG / "test" / "images" / "*")):
    r = model.predict(cale, conf=0.25, imgsz=setari["imgsz"], device=DISPOZITIV, verbose=False)[0]
    if r.masks is None:
        continue
    for cls, chenar, contur in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist(), r.masks.xy):
        arie_chenar = (chenar[2] - chenar[0]) * (chenar[3] - chenar[1])
        if arie_chenar > 0 and len(contur) >= 3:
            rapoarte.setdefault(nume[int(cls)], []).append(cv2.contourArea(contur.astype(np.float32)) / arie_chenar)
raport_arie = {c: {"nr_detectii": len(v), "mediana_masca_pe_dreptunghi": f3(np.median(v))} for c, v in rapoarte.items()}
display(pd.DataFrame(raport_arie).T)

## 7. Salvare model și rezultat (citit de `ruleaza_toate.py`)

In [ ]:
shutil.copy(DOSAR / "weights" / "best.pt", MODELE / f"{NUME_EXPERIMENT}.pt")
epoci_rulate = len(pd.read_csv(DOSAR / "results.csv")) if (DOSAR / "results.csv").exists() else None
rezultat = {
    "experiment": NUME_EXPERIMENT, "stare": "gata", "descriere": DESCRIERE,
    "model": MODEL, "epoci_maxime": setari["epochs"], "epoci_rulate": epoci_rulate,
    "parametri": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                  for k, v in setari.items() if k not in ("augmentations", "project", "name", "data", "exist_ok")},
    "albumentations": [str(t) for t in (AUGMENTARI or [])],
    "metrici": metrici, "arie_masca_vs_dreptunghi": raport_arie,
    "terminat_la": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(rezultat, open(LOGURI / f"{NUME_EXPERIMENT}.json", "w"), indent=2, ensure_ascii=False)
v = metrici["validare"]
print(f"✅ {NUME_EXPERIMENT}: validare Mask mAP50-95 = {v.get('mask', v['box'])['mAP50-95']:.3f} | "
      f"Box mAP50 = {v['box']['mAP50']:.3f}")
print("Model salvat:", MODELE / f"{NUME_EXPERIMENT}.pt")